In [ ]:
import os
import math
import random
import numpy as np
import qiskit
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from qiskit.visualization import plot_bloch_multivector
from qiskit.quantum_info import Statevector
from qiskit import QuantumCircuit
from qiskit.circuit.library import MCMTGate, ZGate
from qiskit.visualization import plot_distribution
from qiskit import transpile
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

token = os.getenv('IBM_QUANTUM_TOKEN')
instance = os.getenv('IBM_QUANTUM_INSTANCE')

try:
    QiskitRuntimeService.save_account(
        channel="ibm_quantum_platform",
        token=token,
        instance=instance,
        overwrite=True)
except Exception as e:
    print(f"Error: {e}")

[Grover's algorithm](https://quantum.cloud.ibm.com/learning/en/modules/computer-science/grovers)

In [ ]:
def generate_states(n):
    num_marked = random.randint(1, (2**n) // 4)
    
    marked_states = []
    while len(marked_states) < num_marked:
        state = "".join(random.choice("01") for _ in range(n))
        if state not in marked_states:
            marked_states.append(state)
    return marked_states

def grover_oracle():
    """Build a Grover oracle for multiple marked states
 
    Here we assume all input marked states have the same number of bits
 
    Parameters:
        marked_states (str or list): Marked states of oracle
 
    Returns:
        QuantumCircuit: Quantum circuit representing Grover oracle
    """
    num_qubits = 4
    marked_states = generate_states(num_qubits)
    print(f"Generated marked states:")
    for state in marked_states:
        print(f"|{state}>")
    
    # Compute the number of qubits in circuit
    num_qubits = len(marked_states[0])
 
    qc = QuantumCircuit(num_qubits)
    # Mark each target state in the input list
    for target in marked_states:
        # Flip target bitstring to match Qiskit bit-ordering
        rev_target = target[::-1]
        # Find the indices of all the '0' elements in bitstring
        zero_inds = [
            ind for ind in range(num_qubits) if rev_target[ind] == "0"
        ]
        # Add a multi-controlled Z-gate with pre- and post-applied X-gates (open-controls)
        # where the target bitstring has a '0' entry
        if zero_inds:
            qc.x(zero_inds)

        qc.compose(MCMTGate(ZGate(), num_qubits - 1, 1), qubits=range(num_qubits), inplace=True)
        
        if zero_inds:
            qc.x(zero_inds)
    return qc.to_gate(label="Oracle")

In [ ]:
def diffuser(n):
    qc = QuantumCircuit(n)
    
    qc.h(range(n))
    
    qc.x(range(n))
    qc.append(MCMTGate(ZGate(), n-1, 1), range(n))
    qc.x(range(n))
    
    qc.h(range(n))
    
    return qc.to_gate(label="Diffuser")

# Why is $Z_{OR}$ implemented as $X$ — $MCMT$ — $X$?

In Grover's Algorithm, the diffuser requires an operator that performs a reflection about the zero state $|0\rangle^{\otimes n}$.

### 1. The Limitation of Standard Control Gates
Most multi-controlled gates (like Qiskit's `MCMTGate`) are designed to flip the phase of the $|11\dots1\rangle$ state.
* **The Goal**: We need to target the $|00\dots0\rangle$ state.
* **The Problem**: A standard Multi-Controlled Z (MCZ) gate will only affect the state where all qubits are $1$.

### 2. X Gates as a State Translator
To bridge this gap, we use $X$ gates (bit-flips) as a temporary "translator":
* **Pre-processing**: Applying $X$ gates to all qubits transforms $|00\dots0\rangle$ into $|11\dots1\rangle$.
* **The Reflection**: The $MCMT$ (Z-logic) now correctly identifies the state and inverts its phase.
* **Post-processing**: Applying $X$ gates again reverts the qubits to their original basis, ensuring only the original $|0\rangle^{\otimes n}$ state was affected.

### 3. Global Phase and Mathematical Equivalence
This implementation inverts the phase of the $|0\rangle^{\otimes n}$ state while leaving all other basis states unchanged. While the theoretical formula often suggests inverting everything *except* the zero state, the difference is merely a **global phase of $-1$**. In quantum mechanics, a global phase does not change the observable probabilities, so the algorithm remains perfectly functional.

In [ ]:
def run_grover(oracle):
    n = oracle.num_qubits 
    
    qc = QuantumCircuit(n)
    
    qc.h(range(n)) # Create even superposition of all basis states

    #N = 2**n
    #A = 1 # We don't know how many marked states there are
    #theta = math.asin(math.sqrt(A / N))
    #t = math.floor(math.pi / (4 * theta))
    t = 1 # Fixed at 1 to prevent 'overcooking' (over-rotation) in small registers (n=3/4) when the exact number of marked states is unknown.
    
    for _ in range(t): # Apply Grover iterations the optimal number of times
        qc.append(oracle, range(n))
        qc.append(diffuser(n), range(n))

    print("Bloch sphere:")
    display(plot_bloch_multivector(Statevector.from_instruction(qc)))
    
    print("Diffuser:")
    display(diffuser(n).definition.draw(output="mpl", style="iqp"))

    qc.measure_all()
    
    return qc

In [ ]:
oracle = grover_oracle()

print("Oracle:")
display(oracle.definition.draw(output="mpl", style="iqp"))

final_circuit = run_grover(oracle)

print("Circuit:")
final_circuit.draw("mpl")

If you want to run the circuit on the Aer simulator:

In [ ]:
shots = 1024

backend = AerSimulator()
job = backend.run(transpile(final_circuit, backend), shots=shots)
result = job.result()
counts = result.get_counts()

If you want to run the circuit on a real quantum computer:

In [ ]:
shots = 10000

service = QiskitRuntimeService()
print("Selecting backend...")
backend = service.least_busy(simulator=False, operational=True)
print(f"Running on backend: {backend.name}")

pm = generate_preset_pass_manager(optimization_level=3, target=backend.target)
 
transpiled_circuit = pm.run(final_circuit)

sampler = Sampler(backend)

job = sampler.run([transpiled_circuit], shots=shots)

job_id = job.job_id()
print(f"Job sumbitted. ID: {job_id}") # Wait for the job to complete and then retrieve the results

Retrieve the results from the job run on the real quantum computer:

In [ ]:
retrieved_job = service.job(job_id)
current_status = retrieved_job.status()
if current_status == 'DONE':
    result = retrieved_job.result()
    counts = result[0].data.meas.get_counts()
    print("Job completed successfully!")
else:
    print(f"Job is not completed yet. Current status: {current_status}")

In [ ]:
probabilities = {state : count / shots for state, count in counts.items()}
print(f"Probabilities {shots} shots:")
display(plot_histogram(probabilities))

print("The marked states are:")
least_prob = min(probabilities.values())
highest_prob = max(probabilities.values())
threshold = (least_prob + highest_prob) / 2
for state, prob in probabilities.items():
	if prob > threshold or highest_prob - least_prob < 0.05:
		print(f"|{state}>")

## Quantum vs Classical Search

### 1. Speedup: O(N) vs O($\sqrt{N}$)
In a classical unsorted database of size **N**, finding all marked items requires checking every element one by one, leading to **N** operations in the worst case. Grover's algorithm reduces this to approximately **$\sqrt{N/k}$** operations, where **k** is the number of marked states (solutions). This quadratic speedup allows quantum computers to search large datasets much faster than classical ones.



### 2. Mechanism: Amplitude Amplification
The algorithm uses quantum interference to highlight all solutions simultaneously. The **Oracle** flips the sign of the **k** marked states, and the **Diffuser** reflects all amplitudes about the average. This process boosts the probability amplitude of all correct states while suppressing the wrong ones. Because probability is the square of the amplitude, the marked states become clearly visible in the final measurement after very few steps.